# Block 2B: NLP — RAG-based Insurance Report Generation

**Goal:** Build a Retrieval-Augmented Generation (RAG) system that uses real NHTSA vehicle
complaint data as a knowledge base and generates structured insurance reports with GPT-4o-mini.

**Pipeline:**
1. Download NHTSA ODI complaint data via public API
2. Build FAISS vector index with sentence-transformers (`all-MiniLM-L6-v2`)
3. At inference: retrieve relevant complaints → prompt GPT-4o-mini → structured JSON report

**Output:** `nhtsa_faiss.index` + `nhtsa_complaints_processed.csv` used by the Streamlit app

## Setup

In [ ]:
import os
import json
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv('../.env')
API_KEY = os.getenv('OPENAI_API_KEY', '')

MODELS_DIR    = Path('../models')
PROCESSED_DIR = Path('../data/processed')
MODELS_DIR.mkdir(exist_ok=True)
PROCESSED_DIR.mkdir(exist_ok=True)

EMBED_MODEL = 'all-MiniLM-L6-v2'
print("Setup complete. API key present:", bool(API_KEY))


## Data Source

**NHTSA ODI Complaints API** — [api.nhtsa.gov](https://api.nhtsa.gov)

We query complaints for common vehicle makes/models. Each complaint contains:
- `cdescr`: free-text description of the complaint
- `compdesc`: component involved (e.g. BODY, EXTERIOR LIGHTING)
- `malfunctionlampon`, `crash`, `fire`, `injury` flags

We filter for body/exterior complaints relevant to physical damage assessment.

## Download NHTSA Complaints

In [ ]:
VEHICLES = [
    ("TOYOTA", "COROLLA", 2020), ("TOYOTA", "CAMRY", 2019),
    ("HONDA", "CIVIC", 2020),   ("HONDA", "CR-V", 2019),
    ("FORD", "F-150", 2020),    ("FORD", "ESCAPE", 2019),
    ("BMW", "3+SERIES", 2019),  ("BMW", "X5", 2020),
    ("VOLKSWAGEN", "GOLF", 2019), ("VOLKSWAGEN", "TIGUAN", 2020),
]

BODY_KEYWORDS = [
    "body", "paint", "scratch", "dent", "rust", "corrosion",
    "bumper", "hood", "door", "panel", "glass", "windshield",
    "headlight", "taillight", "mirror", "fender", "roof",
    "crack", "chip", "damage", "collision", "repair"
]

def fetch_complaints(make, model, year):
    url = f"https://api.nhtsa.gov/complaints/complaintsByVehicle?make={make}&model={model}&modelYear={year}"
    try:
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            return r.json().get("results", [])
    except Exception:
        pass
    return []

def is_relevant(complaint):
    text = (complaint.get("cdescr", "") + " " + complaint.get("compdesc", "")).lower()
    return any(kw in text for kw in BODY_KEYWORDS)

all_complaints = []
for make, model, year in VEHICLES:
    print(f"Fetching {make} {model} {year}...", end=" ")
    results = fetch_complaints(make, model, year)
    relevant = [c for c in results if is_relevant(c)]
    print(f"{len(relevant)}/{len(results)} relevant")
    all_complaints.extend(relevant)
    time.sleep(0.3)

print(f"\nTotal complaints collected: {len(all_complaints)}")


## Process & Clean Text

In [ ]:
def build_text(c):
    comp = c.get("compdesc", "Unknown component")
    desc = c.get("cdescr", "").strip()
    make = c.get("make", "")
    model = c.get("model", "")
    year = c.get("modelYear", "")
    return f"Vehicle: {year} {make} {model} | Component: {comp} | Complaint: {desc[:600]}"

rows = []
for c in all_complaints:
    text = build_text(c)
    if len(text) > 80:
        rows.append({"text": text, "make": c.get("make",""), "model": c.get("model",""),
                     "year": c.get("modelYear",""), "component": c.get("compdesc","")})

df = pd.DataFrame(rows).drop_duplicates(subset='text').reset_index(drop=True)
print(f"Processed: {len(df)} unique complaint texts")
df.head(3)


In [ ]:
# If API returned too few results, add fallback synthetic complaints
MIN_DOCS = 50
if len(df) < MIN_DOCS:
    print(f"Only {len(df)} docs from API, adding synthetic fallback data...")
    synthetic = [
        {"text": f"Vehicle: 2020 TOYOTA COROLLA | Component: BODY | Complaint: Customer reports deep scratch on driver door after minor parking lot incident. Paint down to metal. Requires sanding, priming, and repainting.", "make": "TOYOTA", "model": "COROLLA", "year": 2020, "component": "BODY"},
        {"text": f"Vehicle: 2019 HONDA CIVIC | Component: EXTERIOR LIGHTING | Complaint: Cracked headlight assembly after front collision. Water intrusion causing condensation.", "make": "HONDA", "model": "CIVIC", "year": 2019, "component": "EXTERIOR LIGHTING"},
        {"text": f"Vehicle: 2020 FORD F-150 | Component: BODY | Complaint: Significant dent on rear bumper after low-speed rear-end collision. Bumper cover cracked and needs replacement.", "make": "FORD", "model": "F-150", "year": 2020, "component": "BODY"},
        {"text": f"Vehicle: 2019 BMW 3 SERIES | Component: WINDSHIELD | Complaint: Windshield cracked from small stone impact. Crack spreading across driver field of vision.", "make": "BMW", "model": "3 SERIES", "year": 2019, "component": "WINDSHIELD"},
        {"text": f"Vehicle: 2020 VOLKSWAGEN TIGUAN | Component: BODY | Complaint: Multiple panels show rust spots after 18 months. Paint bubbling on hood and roof.", "make": "VOLKSWAGEN", "model": "TIGUAN", "year": 2020, "component": "BODY"},
        {"text": f"Vehicle: 2019 TOYOTA CAMRY | Component: BODY | Complaint: Door panel torn after accident. Missing trim pieces. Requires full door shell replacement.", "make": "TOYOTA", "model": "CAMRY", "year": 2019, "component": "BODY"},
        {"text": f"Vehicle: 2020 HONDA CR-V | Component: BODY | Complaint: Hood punctured by road debris on highway. Metal bent inward. Engine bay exposed.", "make": "HONDA", "model": "CR-V", "year": 2020, "component": "BODY"},
        {"text": f"Vehicle: 2019 FORD ESCAPE | Component: EXTERIOR LIGHTING | Complaint: Brake light broken in rear collision. Plastic shattered, wiring damaged.", "make": "FORD", "model": "ESCAPE", "year": 2019, "component": "EXTERIOR LIGHTING"},
        {"text": f"Vehicle: 2020 BMW X5 | Component: GLASS | Complaint: Rear windshield shattered spontaneously while parked. No impact observed.", "make": "BMW", "model": "X5", "year": 2020, "component": "GLASS"},
        {"text": f"Vehicle: 2019 VOLKSWAGEN GOLF | Component: BODY | Complaint: Lost front bumper cover on highway due to poor factory clip retention.", "make": "VOLKSWAGEN", "model": "GOLF", "year": 2019, "component": "BODY"},
    ] * 6
    df_synth = pd.DataFrame(synthetic).drop_duplicates(subset='text')
    df = pd.concat([df, df_synth], ignore_index=True)
    print(f"Total after fallback: {len(df)}")

print(f"Final corpus size: {len(df)} documents")


## Build FAISS Vector Index

In [ ]:
print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)

print("Encoding documents...")
embeddings = embedder.encode(df['text'].tolist(), batch_size=32, show_progress_bar=True)
embeddings = embeddings.astype('float32')
faiss.normalize_L2(embeddings)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # Inner product = cosine similarity after L2 norm
index.add(embeddings)

print(f"Index built: {index.ntotal} vectors, dim={dim}")


## Test Retrieval

In [ ]:
def retrieve(query, k=3):
    q_emb = embedder.encode([query]).astype('float32')
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, k)
    return [(df['text'].iloc[i][:300], float(scores[0][j])) for j, i in enumerate(indices[0])]

results = retrieve("broken glass windshield Toyota")
print("Query: broken glass windshield Toyota\n")
for i, (text, score) in enumerate(results):
    print(f"[{i+1}] score={score:.3f}\n{text}\n")


## Test Full Pipeline (RAG + GPT-4o-mini)

In [ ]:
if not API_KEY:
    print("No API key — skipping live test. Set OPENAI_API_KEY in .env")
else:
    client = OpenAI(api_key=API_KEY)

    query = "scratch Toyota front bumper"
    context = '\n---\n'.join([t for t, _ in retrieve(query, k=3)])

    prompt = f"""Generate a structured vehicle damage report as JSON.

VEHICLE: 2020 Toyota Corolla, value $18,000
DAMAGE: scratch on front bumper (AI confidence: 87%)
ESTIMATED COST: $450 (range: $360-$540)

RELEVANT CONTEXT:
{context}

Return this exact JSON:
{{
  \"vehicle\": {{\"make\": \"\", \"model\": \"\", \"year\": 0, \"value_usd\": 0}},
  \"damage\": {{\"type\": \"\", \"location\": \"\", \"severity\": \"\", \"description\": \"\"}},
  \"assessment\": {{\"repair_recommendation\": \"\", \"estimated_cost_usd\": 0, \"cost_range\": \"\", \"confidence_level\": \"\"}},
  \"notes\": \"\"
}}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=700,
        messages=[
            {"role": "system", "content": "You are an insurance claims assessor. Always respond with valid JSON only."},
            {"role": "user", "content": prompt}
        ],
        response_format={"type": "json_object"}
    )
    report = json.loads(response.choices[0].message.content)
    print("Generated report:")
    print(json.dumps(report, indent=2))


## Corpus Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['component'].value_counts().head(10).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 10 Components in Knowledge Base')
axes[0].set_xlabel('Count')

df['make'].value_counts().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Complaints by Vehicle Make')
axes[1].set_xlabel('Make')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'nlp_corpus_analysis.png', dpi=150)
plt.show()

df['text_len'] = df['text'].str.len()
print(f'Text length — mean: {df["text_len"].mean():.0f} | min: {df["text_len"].min()} | max: {df["text_len"].max()}')


## Save Artifacts

In [ ]:
# Save FAISS index
faiss.write_index(index, str(MODELS_DIR / 'nhtsa_faiss.index'))

# Save processed complaints CSV
df.to_csv(PROCESSED_DIR / 'nhtsa_complaints_processed.csv', index=False)

print(f"Saved: nhtsa_faiss.index ({index.ntotal} vectors)")
print(f"Saved: nhtsa_complaints_processed.csv ({len(df)} rows)")


## Summary

| Artifact | Description |
|---|---|
| `nhtsa_faiss.index` | FAISS cosine-similarity index, 384-dim embeddings |
| `nhtsa_complaints_processed.csv` | Processed NHTSA complaint texts |
| Embedding model | `all-MiniLM-L6-v2` (sentence-transformers) |
| LLM | `gpt-4o-mini` via OpenAI API |

**RAG flow:** damage query → top-3 similar complaints retrieved → context injected into GPT prompt → structured JSON insurance report.

**Integration:** `src/nlp_report.py` loads these artifacts and is called by the Streamlit app.